##RAG PIPELINE - Data ingestion to vector db pipeline


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs_in_directory(pdf_directory):
    all_documents = []
    pdf_dir =  Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file.name}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            document = loader.load()

            for doc in document:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            
            all_documents.extend(document)
            print(f"  ✓ Loaded {len(document)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs_in_directory("../data")

In [ ]:
all_pdf_documents

###TEXT SPLITTing get into chunks

In [ ]:
def split_documents(document, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(document)
    print(f"Split {len(document)} documents into {len(chunks)} chunks")

    #show example of a chunk
    if chunks:
        print("\nExample chunk:")
        print(f"Content: {chunks[0].page_content[:200]}...")
        print(f"Metadata: {chunks[0].metadata}")
    return chunks  # Return the split document chunks

In [ ]:
chunks = split_documents(all_pdf_documents)
chunks

###EMbedding and vector store DB


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings    
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """"Handles document embedding generation using sentence transformers."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()
    

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of text documents."""
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings =   self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


In [ ]:
##INitialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager


###Vector Store

In [ ]:
class VectorStore:
    """Manages doc embeddings in chromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
    
    def _initialize_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            #create persistent chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            #get or create collection

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
                )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store."""
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        document_texts = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata =dict(doc.metadata) 
            metadata['doc_index'] = i
            metadata ['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #doc content
            document_texts.append(doc.page_content)


            #embedding
            embedding_list.append(embedding.tolist())

        #add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings  = embedding_list,
                metadatas=metadatas,
                documents=document_texts
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


In [ ]:
vector_store = VectorStore()
vector_store

In [ ]:
chunks

#convert the text to embeddings and store in vector store


In [ ]:
texts = [doc.page_content for doc in chunks] 

#generaate embeddings for the chunks
embeddings = embedding_manager.generate_embeddings(texts)

##store in the vector db
vector_store.add_documents(chunks, embeddings)


In [ ]:
class RAGRetriever:
    """handles query based retrieval from the vector store"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """init the retriever
        Arggs: 
        vector store = vector store having doc embeddings
        embedding_manager = manager to generate query embeddings"""
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
            

ragretriever = RAGRetriever(vector_store, embedding_manager)

In [ ]:
ragretriever

In [ ]:
ragretriever.retrieve("When was paleolithic age")

In [ ]:
ragretriever.retrieve("who was Akbar")


In [ ]:
ragretriever.retrieve("When did India gain independence")

In [ ]:
ragretriever.retrieve("When was indus valley civilization discovered?")

In [ ]:
ragretriever.retrieve("Neolithic -Chalcolithic culture of the South")

# INTEGRATION VectorDB context pipeline with LLM output

In [ ]:
import os
print(os.getcwd())

In [ ]:
from langchain_groq import ChatGroq

In [ ]:

##simple RAG pipeline with openAI llm

import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

#initialize the chat model
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model_name="openai/gpt-oss-120b", groq_api_key=groq_api_key, temperature=0.1, max_tokens=1024)

# simple RAG function : retrieve context + generate answer

def rag_simple(query, retriever, llm, top_k=3 ):
    ##retrieve context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "" 
    if not context:
        return "No relevant documents found for the query."
    #generate answer using llm
    prompt = f""" use the the following context to answer the question concisely. 
        Context: {context}
        question: {query}
        Answer:"""
        
    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content
    

In [ ]:
answer = rag_simple("tell me about rig veda", ragretriever, llm)
print(answer)

# Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:600] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("rig veda", ragretriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])